In [1]:
import re

In [2]:
import json

In [3]:
import pickle

In [4]:
import tqdm

In [5]:
from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [7]:
import numpy as np

In [8]:
import pandas as pd

In [9]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [10]:
MODEL_ID = "google/gemma-4-E2B-it"

In [11]:
processor = AutoProcessor.from_pretrained(MODEL_ID, temperature=0.1)

In [12]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [13]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags = pickle.load(f)

In [14]:
data = []

In [15]:
for title, tags in titles_with_tags.items():
    for tag in tags:
        data.append((title, tag))

In [16]:
df = pd.DataFrame(data=data, columns=["title_rus", "tag"])

In [17]:
df.head(10)

,title_rus,tag
0,Исследование приоритетов и механизмов реализац...,international_relations
1,Исследование приоритетов и механизмов реализац...,political_economics
2,Исследование приоритетов и механизмов реализац...,policy_analysis
3,Исследование приоритетов и механизмов реализац...,geopolitics
4,Исследование приоритетов и механизмов реализац...,BRICS
5,Исследование приоритетов и механизмов реализац...,international_development
6,Антрополе - научно-популярный видео-подкаст о ...,anthropology
7,Антрополе - научно-популярный видео-подкаст о ...,social_phenomena
8,Антрополе - научно-популярный видео-подкаст о ...,media
9,Антрополе - научно-популярный видео-подкаст о ...,podcasting


In [18]:
with open("artificial_profiles.json", "r") as f:
    profiles = json.load(f)

In [19]:
profiles

{'international_relations': {'international_relations_specialist': 'A user deeply interested in international relations, with a strong focus on geopolitical analysis, regional studies (especially Eurasia, Asia, Middle East, and Latin America), BRICS dynamics, bilateral relations (Russia-China, Russia-Japan), international law/security issues, and the intersection of politics, economics, and media in global affairs. Highly engaged with academic and extracurricular international forums (Model UN, clubs, conferences).'},
 'sociology': {'sociology_and_cultural_studies_researcher': 'A user deeply interested in sociology, cultural studies, and intercultural communication. They are engaged in qualitative research (interviews, narrative analysis), media studies (film, podcasts, digital spaces), identity formation (ethnic, gender, political), social phenomena, and the intersection of culture, politics, and society. They value theoretical depth and practical applications in areas like social ent

In [20]:
results = {}

In [21]:
for tag, row in list(profiles.items()):
    for profile, description in row.items():
        SYSTEM_PROMPT = f"""
        You are a {profiles}.
        {description}
        Evaluate, how much you would like proposed student project based on its title in russian. Rate it on a scale [None, 1, 2, 3, 4, 5], where the higher the better.
        If proposed project title does not fit your interests, just return None.
        Return stricly JSON.
        For example.
        """
        SYSTEM_PROMPT += """
        ```json
        {'score': 4}
        ```
        """

        messages = []
        scores = {}
        for row in df[df["tag"] == tag].sample(frac=0.25).iterrows():
            messages.append(
                (
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": row[1]["title_rus"]},
                )
            )
        for message in tqdm.tqdm(messages):
            text = processor.apply_chat_template(
                message, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )

            inputs = processor(text=text, return_tensors="pt").to(model.device)
            input_len = inputs["input_ids"].shape[-1]

            outputs = model.generate(**inputs, max_new_tokens=1024)
            response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

            try:
                json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
                if json_match:
                    json_str = json_match.group(1)
                else:
                    json_str = response
                answer = json.loads(json_str)
            except Exception:
                answer = {"score": "None"}

            score = answer.get("score", "None")
            try:
                score = int(score)
                if score > 3:
                    scores[message[1]["content"]] = score
            except Exception:
                pass

        values = list(scores.values())
        if len(values):
            median = np.median(values)
        else:
            median = np.nan

        print(f"{profile}: {len(values)}, {median}")
        results[profile] = scores

100%|██████████| 24/24 [03:14<00:00,  8.10s/it]


international_relations_specialist: 18, 5.0


100%|██████████| 24/24 [03:13<00:00,  8.08s/it]


sociology_and_cultural_studies_researcher: 19, 5.0


100%|██████████| 23/23 [03:00<00:00,  7.86s/it]


...

digital_marketing_strategist: 8, 5.0


100%|██████████| 7/7 [00:54<00:00,  7.78s/it]


journalism_and_media_strategist: 5, 5.0


100%|██████████| 7/7 [00:53<00:00,  7.64s/it]

web_developer_entrepreneur: 2, 5.0


In [22]:
for researcher in results:
    data = [x for x in results[researcher].items() if x[1]]
    if data:
        for topic, score in sorted(data, key=lambda x: x[1], reverse=True)[:3]:
            print(f"{researcher}, {topic}, {score}")
    print()

international_relations_specialist, "Политика Европейского Союза в области прав человека: сравнительный анализ внутреннего принуждения и внешней обусловленности в Венгрии и Грузии	", 5
international_relations_specialist, Влияние США, европейских и ближневосточных государств и Турции на энергетические отношения Китая и России со странами Центральной Азии., 5
international_relations_specialist, Эволюция политических отношений между Пакистаном и Ираном после 1979 года, 5

sociology_and_cultural_studies_researcher, Искусственный интеллект (ИИ) в развитии экономики, политики, социума, 5
sociology_and_cultural_studies_researcher, Сравнительный анализ характеристик работы представителей поколения Z в Нигерии и Англии, 5
sociology_and_cultural_studies_researcher, Социальная теория и этнография: 2025/2026, 5

economics_macro_analyst, Каналы денежной трансмиссии и их особенности в современных условиях, 5
economics_macro_analyst, Оценка роли выпуска субфедеральных облигаций в формировании финансо

In [23]:
with open("artificial_profiles_scores.pkl", "wb") as f:
    pickle.dump(results, f)